* Label Encoding
* One-Hot Encoding
* Feature Scaling
* Create new features
* Remove highly correlated features
* Train/Test split

In [134]:
import pandas as pd
df = pd.read_csv(r'D:\Desktop\mlops\customer-churn-monitoring-mlops\data\processed\cleaned_data.csv')
df.head(3)

,tenure_months,monthly_charges,usage_minutes,support_calls,payment_delay_days,contract_type,autopay,num_services,satisfaction_score,signup_channel,marketing_segment,churn
0,60,78.04,512,8,4.4,two-year,1,3,6.0,referral,B,0
1,6,140.22,170,3,5.4,month-to-month,0,2,5.0,referral,B,0
2,39,89.22,377,1,0.0,month-to-month,0,2,10.0,ads,C,1


In [135]:
X = df.drop('churn', axis = 1)
y = df['churn']

In [136]:
print(X.shape)
y.shape

(11000, 11)


(11000,)

In [137]:
X['signup_channel'].unique()

<StringArray>
['referral', 'ads', 'organic', 'partner']
Length: 4, dtype: str

In [138]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, random_state=42, test_size=0.2)

encoding on categorical features - contract_type, signup_channel, marketing_segment
scaling on numeric features - tenure_months,	monthly_charges,	usage_minutes,	payment_delay_days

In [139]:
print(type(X_train))
print(type(X_test))

print(X_train.shape)
print(X_test.shape)

<class 'pandas.DataFrame'>
<class 'pandas.DataFrame'>
(8800, 11)
(2200, 11)


In [140]:
print(type(X_train))

<class 'pandas.DataFrame'>


In [141]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

categorical_cols = [
    "contract_type",
    "marketing_segment",
    "signup_channel"
]

encoder = OneHotEncoder(
    drop="first",
    handle_unknown="ignore",
    sparse_output=False
)

# Encode categorical columns
X_train_encoded = encoder.fit_transform(X_train[categorical_cols])
X_test_encoded = encoder.transform(X_test[categorical_cols])

# Convert encoded arrays to DataFrames
encoded_cols = encoder.get_feature_names_out(categorical_cols)

X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=encoded_cols,
    index=X_train.index
)

X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=encoded_cols,
    index=X_test.index
)

# Remove original categorical columns
X_train = X_train.drop(columns=categorical_cols)
X_test = X_test.drop(columns=categorical_cols)

# Add encoded columns back
X_train = pd.concat([X_train, X_train_encoded], axis=1)
X_test = pd.concat([X_test, X_test_encoded], axis=1)

In [142]:
X_train

,tenure_months,monthly_charges,usage_minutes,support_calls,payment_delay_days,autopay,num_services,satisfaction_score,contract_type_one-year,contract_type_two-year,marketing_segment_B,marketing_segment_C,signup_channel_organic,signup_channel_partner,signup_channel_referral
10735,1,132.06,723,6,11.5,0,1,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
5937,25,45.39,439,3,13.9,0,1,5.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
7642,7,46.79,643,4,3.2,1,3,6.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3328,23,102.44,423,3,3.6,0,3,6.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
8681,40,27.25,361,7,0.0,0,3,7.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5734,34,79.65,130,3,6.6,1,4,8.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
5191,40,48.55,560,4,12.3,0,4,6.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
5390,13,123.33,467,1,8.4,1,3,6.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
860,63,137.03,618,0,0.0,1,5,5.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [143]:
from sklearn.preprocessing import StandardScaler
import joblib

num_cols = [
    "tenure_months",
    "monthly_charges",
    "usage_minutes",
    "payment_delay_days"
]

scaler = StandardScaler()

# Scale and assign column by column
X_train[num_cols] = pd.DataFrame(
    scaler.fit_transform(X_train[num_cols]),
    columns=num_cols,
    index=X_train.index
)

X_test[num_cols] = pd.DataFrame(
    scaler.transform(X_test[num_cols]),
    columns=num_cols,
    index=X_test.index
)



In [144]:
X_train.head(3)

,tenure_months,monthly_charges,usage_minutes,support_calls,payment_delay_days,autopay,num_services,satisfaction_score,contract_type_one-year,contract_type_two-year,marketing_segment_B,marketing_segment_C,signup_channel_organic,signup_channel_partner,signup_channel_referral
10735,-1.931505,1.253421,2.326287,6,1.657951,0,1,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
5937,-0.607367,-1.073022,0.145557,3,2.237412,0,1,5.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
7642,-1.600471,-1.035443,1.711997,4,-0.346020,1,3,6.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [146]:
import os
import joblib

# Create folders if they don't exist
os.makedirs("models", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# Files to remove if they already exist
files_to_remove = [
    "models/scaler.pkl",
    "models/onehot_encoder.pkl",
    "data/processed/X_train.csv",
    "data/processed/X_test.csv",
    "data/processed/y_train.csv",
    "data/processed/y_test.csv"
]

for file in files_to_remove:
    if os.path.exists(file):
        os.remove(file)

# Save processed datasets
X_train.to_csv("data/processed/X_train.csv", index=False)
X_test.to_csv("data/processed/X_test.csv", index=False)

y_train.to_csv("data/processed/y_train.csv", index=False)
y_test.to_csv("data/processed/y_test.csv", index=False)

# Save preprocessing objects
joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(encoder, "models/onehot_encoder.pkl")

print("All processed data and preprocessing objects saved successfully.")

All processed data and preprocessing objects saved successfully.
